# บทเรียน 01 - บทนำสู่เอเย่นต์ AI

ยินดีต้อนรับสู่บทเรียนแรกในหลักสูตร **AI Agents for Beginners**!

**เอเย่นต์ AI** คือโปรแกรมที่ใช้โมเดลภาษาขนาดใหญ่ (LLM) เป็นเครื่องยนต์ตรรกะและสามารถดำเนิน *การกระทำ* ในโลกจริง — เรียกใช้ API, สอบถามฐานข้อมูล หรือรันโค้ด — เพื่อบรรลุเป้าหมายในนามของผู้ใช้

ในโน้ตบุ๊กนี้คุณจะสร้างเอเย่นต์ตัวแรกของคุณ: **Travel Agent** ที่แนะนำจุดหมายปลายทางสำหรับวันหยุด ในระหว่างทางคุณจะได้เรียนรู้วิธี:

1. เชื่อมต่อกับ Microsoft Foundry Agent Service โดยใช้ **Microsoft Agent Framework**  
2. ให้เอเย่นต์มี **เครื่องมือ** — ฟังก์ชัน Python ธรรมดาที่มันสามารถเรียกใช้ได้  
3. รันเอเย่นต์และตรวจสอบคำตอบของมัน  
4. สตรีมคำตอบของเอเย่นต์ทีละโทเคน  


## การตั้งค่า

ก่อนที่จะรันโน้ตบุ๊กนี้ โปรดตรวจสอบให้แน่ใจว่าคุณมี:

1. **โปรเจกต์ Microsoft Foundry** ที่มีโมเดลแชทที่ได้ถูกดีพลอยแล้ว (เช่น `gpt-5-mini`)
2. **เข้าสู่ระบบด้วย Azure CLI** — รันคำสั่ง `az login` ในเทอร์มินอลของคุณ
3. **ตั้งค่าตัวแปรสภาพแวดล้อมที่จำเป็น:**
   - `AZURE_AI_PROJECT_ENDPOINT` — จุดสิ้นสุดของโปรเจกต์ Microsoft Foundry ของคุณ
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ชื่อของโมเดลที่คุณได้ดีพลอย

เซลล์ด้านล่างนี้ติดตั้งแพ็กเกจ Python ที่คุณต้องการ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## การสร้างเอเย่นต์ตัวแรกของคุณ

เอเย่นต์ต้องการสองสิ่ง:

- **คำสั่ง** ที่บอกว่า *มันคือใคร* และ *ควรประพฤติตัวอย่างไร* (พรอมต์ระบบ)
- **เครื่องมือ** — ฟังก์ชัน Python ที่ตกแต่งด้วย `@tool` ซึ่งเอเย่นต์สามารถเรียกใช้เพื่อดึงข้อมูลหรือดำเนินการบางอย่าง

ด้านล่างนี้เรากำหนดเครื่องมืออย่างง่ายที่ส่งกลับรายการสถานที่ท่องเที่ยวยอดนิยม เอเย่นต์จะใช้เครื่องมือนี้เมื่อผู้ใช้ขอคำแนะนำการเดินทาง


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## การตอบสนองแบบสตรีม

สำหรับประสบการณ์ที่มีปฏิสัมพันธ์มากขึ้น คุณสามารถ **สตรีม** การตอบสนองของเอเจนต์ได้ แทนที่จะรอการตอบกลับทั้งหมด เอเจนต์จะส่งข้อความเป็นส่วนๆ เมื่อมันถูกสร้างขึ้น วิธีนี้มีประโยชน์อย่างยิ่งในอินเทอร์เฟซแชทที่คุณต้องการแสดงผลแบบเรียลไทม์


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## สรุป

ในบทเรียนนี้คุณได้เรียนรู้วิธีการ:

- **สร้างโปรไวเดอร์** ที่เชื่อมต่อกับ Microsoft Foundry Agent Service ผ่าน `FoundryChatClient`
- **กำหนดเครื่องมือ** โดยใช้ตัวตกแต่ง `@tool` เพื่อให้อีเจนต์สามารถเรียกใช้ฟังก์ชัน Python ของคุณ
- **รันอีเจนต์** ด้วยข้อความของผู้ใช้และพิมพ์การตอบกลับของมัน
- **ส่งมอบการตอบกลับแบบสตรีม** สำหรับผลลัพธ์แบบเรียลไทม์

ในบทเรียนถัดไป เราจะสำรวจกรอบงานอีเจนต์ในเชิงลึกมากขึ้นและเรียนรู้วิธีมอบเครื่องมือที่ทรงพลังยิ่งขึ้นและความสามารถในการให้เหตุผลแบบหลายขั้นตอนแก่อีเจนต์


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
